# RF electrode of a silicon modulator

A silicon traveling-wave modulator is driven through a pair of metal strips that carry the RF signal along the phase shifter. How much of the signal reaches the end of the strips, and at what impedance, sets how fast the modulator can be driven. This guide simulates that electrode, a coplanar-strip line on a silicon-on-insulator wafer, and compares it with the measurement of Baehr-Jones *et al.*, *Optics Express* **20**(11), 12014 (2012).

It is simulated with GDSFactory RF simulation, based on [Palace](https://github.com/awslabs/palace) (finite elements, run in the cloud), and cross-checked with femwell, a second finite-element solver that runs on a laptop.

> **Result at 10 GHz**
>
> - Bulk aluminum gives **1.83 dB/cm** of loss, about half of the measured **3.8 dB/cm**: the model's strip resistance is too low, not its inductance or capacitance.
> - Palace with the aluminum meshed as a volume and femwell agree on the loss of bulk aluminum (**1.826 dB/cm** from both, same mesh) and on the impedance within 0.2 %.
> - Palace's surface-impedance condition for the metal gives **1.40 dB/cm**, 23 % under femwell's loss for the same metal.
> - With one conductivity fitted to the paper's resistance, a 3D Palace model gives **3.56 dB/cm** and Z = 58.1 - 5.2j ohm. That agrees with the paper because it was fitted to it: a calibration, not a prediction.
>
> The Palace numbers are from cloud runs. The stored results record the Palace version (a build that predates the wave-port impedance fix awslabs/palace#1119) but not the hardware, and the table says what is not recorded. This guide recomputes everything that runs on a laptop.

The long version, with every job, the fit and the impedance definitions, is [RF electrode of a silicon modulator: simulation details](palace_bj2012_electrode_details.md).

## The device

The paper's modulator puts a lateral PN junction in a silicon rib waveguide and drives it with a travelling-wave electrode. Its "metal-only" control line is the same electrode without the junction. Measured at 10 GHz it has $Z = 61 - 5j\,\Omega$, a loss of 3.8 dB/cm, $L = 457$ nH/m, $R = 4.9$ k$\Omega$/m and $C = 0.12$ nF/m (Baehr-Jones *et al.*, [doi:10.1364/OE.20.012014](https://doi.org/10.1364/OE.20.012014)).

The cross-section is read from the paper's Fig. 1b: two aluminum strips, 50 µm wide and 2 µm thick, with a 5 µm gap, on 2 µm of buried oxide (BOX) over a 750 $\Omega$·cm silicon handle wafer. Four inputs are assumptions, because the paper does not give them:

- the strips' lower face is 1.5 µm above the BOX;
- the top silicon under the line is absent;
- no oxide covers the strips;
- the handle wafer is truncated at 100 µm, and the domain is ±200 µm wide with 100 µm of air above, inside a perfect-conductor box (which is also what the wave port's own solve assumes).

The numbers of the paper (read from its text and, for the figures, by eye) are in `data/bj2012_reference.json`; every number this guide computes or quotes is in `data/rf_benchmark/bj2012_electrode.json`.

In [ ]:
import json
import logging
from pathlib import Path

import gdsfactory as gf
import matplotlib.pyplot as plt
import meshio
import numpy as np
from IPython.display import Markdown, display
from matplotlib.collections import PolyCollection
from scipy.constants import speed_of_light

from gsim.common.modes import Conductor, propagating_modes
from gsim.common.stack.extractor import Layer, LayerStack
from gsim.common.stack.materials import MaterialProperties
from gsim.common.transmission_line import line_params_from_neff
from gsim.femwell import epsilon_by_region, line_reading, solve_modes
from gsim.palace import BoundaryModeSim, DrivenSim

# The mesher's advice on air boxes targets other simulations.
logging.getLogger("gsim.palace").setLevel(logging.ERROR)

FREQ_HZ = 10e9  # Hz, the paper's frequency

REF_PATH = next(
    p
    for p in (
        Path("data/bj2012_reference.json"),
        Path("nbs/data/bj2012_reference.json"),
    )
    if p.exists()
)
REF = json.loads(REF_PATH.read_text(encoding="utf-8"))
PAPER = REF["metal_only_line_10GHz"]
STORED = json.loads(
    (REF_PATH.parent / "rf_benchmark" / "bj2012_electrode.json").read_text(
        encoding="utf-8"
    )
)
ROWS = {row["id"]: row for row in STORED["rows"]}

In [ ]:
STRIP_WIDTH = REF["geometry"]["electrode_strip_width_um"]  # um
GAP = REF["geometry"]["electrode_gap_um"]  # um
METAL_THICKNESS = REF["geometry"]["top_metal_thickness_um"]  # um
BOX_THICKNESS = REF["geometry"]["box_thickness_um"]  # um
HANDLE_DEPTH = 100.0  # um, the handle wafer is truncated here
HALF_WIDTH = 200.0  # um, half-width of the domain
AIR_ABOVE = 100.0  # um
METAL_BOTTOM = 1.5  # um, strips' lower face above the BOX
OXIDE_OVER_METAL = 0.0  # um, no passivation over the top aluminum
SIGMA_AL_BULK = 3.77e7  # S/m, bulk aluminum
SIGMA_AL = 1.0e7  # S/m, effective: femwell's R at 10 GHz is the paper's 4.9 kohm/m
SIGMA_HANDLE = 100.0 / REF["geometry"]["handle_resistivity_ohm_cm"]  # S/m
STRIPS = {  # name: (y0, y1) in um
    "strip_g": (-GAP / 2 - STRIP_WIDTH, -GAP / 2),
    "strip_s": (GAP / 2, GAP / 2 + STRIP_WIDTH),
}
STRIP_Z = (METAL_BOTTOM, METAL_BOTTOM + METAL_THICKNESS)


def build_cps(length_um=10.0, *, lossy_metal=False, ports=False, sigma=SIGMA_AL):
    """Component and layer stack of the CPS (strips along x, ``length_um`` long).

    ``lossy_metal`` meshes the strips as regions of aluminum (for femwell);
    otherwise they are conductor layers: outlines in 2D, finite-conductivity surfaces
    in 3D. ``ports`` puts a port at each end on the signal strip. ``sigma`` is the
    strips' conductivity (S/m).
    """
    gf.gpdk.PDK.activate()
    comp = gf.Component()
    layers = {}

    def add(name, gds_layer, y0, y1, z0, z1, material, layer_type):
        comp.add_polygon(
            [(0, y0), (length_um, y0), (length_um, y1), (0, y1)], layer=gds_layer
        )
        layers[name] = Layer(
            name=name,
            gds_layer=gds_layer,
            zmin=z0,
            zmax=z1,
            thickness=z1 - z0,
            material=material,
            layer_type=layer_type,
            mesh_resolution=1.0,
        )

    add("domain", (99, 0), -HALF_WIDTH, HALF_WIDTH, -1.0, -0.9, "sio2_rf", "dielectric")
    for i, (name, span) in enumerate(STRIPS.items()):
        add(
            name,
            (10 + i, 0),
            *span,
            *STRIP_Z,
            "aluminum",
            "dielectric" if lossy_metal else "conductor",
        )
    if ports:
        for name, x, orientation in (("o1", 0.0, 180), ("o2", length_um, 0)):
            comp.add_port(
                name=name,
                center=(x, sum(STRIPS["strip_s"]) / 2),
                width=STRIP_WIDTH,
                orientation=orientation,
                port_type="electrical",
                layer=(11, 0),
            )
    z_oxide_top = STRIP_Z[1] + OXIDE_OVER_METAL
    stack = LayerStack(
        pdk_name="bj2012_cps",
        layers=layers,
        materials={
            "sio2_rf": MaterialProperties(permittivity=3.9).to_dict(),
            "handle_si": MaterialProperties(
                permittivity=11.9, conductivity=SIGMA_HANDLE
            ).to_dict(),
            "aluminum": MaterialProperties(
                permittivity=1.0, conductivity=sigma
            ).to_dict(),
            "air": MaterialProperties(permittivity=1.0).to_dict(),
        },
        dielectrics=[
            {
                "name": "handle",
                "zmin": -BOX_THICKNESS - HANDLE_DEPTH,
                "zmax": -BOX_THICKNESS,
                "material": "handle_si",
            },
            {"name": "box", "zmin": -BOX_THICKNESS, "zmax": 0.0, "material": "sio2_rf"},
            {"name": "oxide", "zmin": 0.0, "zmax": z_oxide_top, "material": "sio2_rf"},
        ],
    )
    return comp, stack


def conductors(model):
    """The signal and ground strips as the current integrals name them."""
    return (
        Conductor("strip_s", (STRIPS["strip_s"], STRIP_Z), model),
        Conductor("strip_g", (STRIPS["strip_g"], STRIP_Z), model),
    )


# Refinement box in the gap and around the strip edges: (y0, y1, z0, z1, size) um.
GAP_BOX = (-10.0, 10.0, 0.8, 5.0, 0.5)


def cross_section_sim(
    lossy_metal, output_dir, *, refined_um=1.0, metal_mesh_um=1.0, sigma=SIGMA_AL
):
    """A meshed `BoundaryModeSim` of the CPS cross-section at x = 5 um."""
    comp, stack = build_cps(lossy_metal=lossy_metal, sigma=sigma)
    for name in STRIPS:
        stack.layers[name].mesh_resolution = metal_mesh_um
    sim = BoundaryModeSim()
    sim.set_output_dir(output_dir)
    sim.set_stack(stack)
    sim.set_geometry(comp)
    sim.set_airbox(margin_x=0.0, margin_y=0.0, z_above=AIR_ABOVE, material="air")
    sim.set_cross_section("x=5")
    sim.mesh(
        preset="coarse",
        refined_mesh_size=refined_um,
        max_mesh_size=20.0,
        refinement_boxes=[GAP_BOX],
        verbose=False,
    )
    return sim, stack

In [ ]:
# The 2D job of Palace that meshes the strips as aluminum (J1vb in the details),
# and the mesh that femwell solves below.
def boundary_mode_sim(freq_ghz, *, volume_metal=False, sigma=SIGMA_AL, tag=""):
    """J1 family: the 2D solve at one frequency, with the voltage path across the gap.

    ``volume_metal`` meshes the strips as regions of metal of conductivity ``sigma``
    instead of leaving them as outlines with a surface condition.
    """
    sim, _stack = cross_section_sim(
        volume_metal,
        f"./palace-sim-bj2012-2d{tag}-{freq_ghz:g}ghz",
        metal_mesh_um=0.25 if volume_metal else 1.0,
        sigma=sigma,
    )
    sim.metallic_boundaries = True  # the same perfect-conductor wall as the port solve
    sim.solver.linear.mg_max_levels = 1  # multigrid off: awslabs/palace#1177
    sim.set_boundary_mode(
        freq=freq_ghz * 1e9, num_modes=4, save=0, target=2.3, tolerance=1e-8
    )
    z_mid = sum(STRIP_Z) / 2
    inset = 1e-3 * min(GAP, METAL_THICKNESS)  # a hair inside the gap, on the domain
    sim.add_impedance_path(
        "gap",
        voltage=[[GAP / 2 - inset, z_mid], [-(GAP / 2 - inset), z_mid]],
    )
    sim.write_config()
    return sim


sim_2d = boundary_mode_sim(10.0, volume_metal=True, sigma=SIGMA_AL_BULK, tag="-volbulk")
mesh_2d = meshio.read(sim_2d.mesh_path, file_format="gmsh")
print(
    f"2D mesh: {len(mesh_2d.cells_dict['triangle'])} triangles, "
    f"regions {sorted(n for n, (_t, d) in mesh_2d.field_data.items() if d == 2)}"
)

In [ ]:
REGION_LABELS = {  # mesh region -> what it is
    "handle_si": "silicon handle (750 ohm cm)",
    "sio2_rf": "oxide: BOX and fill",
    "strip_g": "ground strip",
    "strip_s": "signal strip",
    "air": "air",
    "domain": "oxide slice",
}
REGION_COLORS = {
    "handle_si": "#c9d6e8",
    "sio2_rf": "#f2d9a6",
    "strip_g": "#7f7f7f",
    "strip_s": "#d62728",
    "air": "#ffffff",
    "domain": "#f2d9a6",
}


def plot_cross_section(mesh, *, ax, title=None, xlim=None, ylim=None):
    """Draw a 2D mesh with one colour per region (the triangles' edges are drawn)."""
    points = mesh.points[:, :2]
    names = {tag: name for name, (tag, dim) in mesh.field_data.items() if dim == 2}
    triangles = mesh.cells_dict["triangle"]
    tags = mesh.cell_data_dict["gmsh:physical"]["triangle"]
    for tag in sorted(set(tags)):
        ax.add_collection(
            PolyCollection(
                points[triangles[tags == tag]],
                rasterized=True,  # thousands of triangles: keep the figure small
                facecolors=REGION_COLORS.get(names[tag], "#bbbbbb"),
                edgecolors="k",
                linewidths=0.08,
            )
        )
    ax.autoscale_view()
    ax.set_aspect("equal")
    if xlim is not None:
        ax.set_xlim(xlim)
    if ylim is not None:
        ax.set_ylim(ylim)
    ax.set(xlabel="y (um)", ylabel="z (um)", title=title)
    return ax


fig, axes = plt.subplots(2, 1, figsize=(11, 7.5), height_ratios=(1.3, 1))
plot_cross_section(
    mesh_2d, ax=axes[0], title="Whole domain (the box of the simulation)"
)
plot_cross_section(
    mesh_2d,
    ax=axes[1],
    title="Around the strips: the mesh Palace and femwell solve on",
    xlim=(-60, 60),
    ylim=(-5, 8),
)
for name, (y0, y1) in STRIPS.items():
    axes[1].text(
        (y0 + y1) / 2,
        sum(STRIP_Z) / 2,
        "signal" if name == "strip_s" else "ground",
        ha="center",
        va="center",
        fontsize=8,
        color="w",
    )
axes[1].text(0, 5.5, "air", ha="center", fontsize=8)
axes[1].text(0, -1.0, "BOX (2 um)", ha="center", fontsize=8)
axes[1].text(-45, -3.8, "silicon handle", ha="center", fontsize=8)
axes[1].text(0, 0.75, "oxide", ha="center", fontsize=8)
plt.tight_layout()
plt.show()

In [ ]:
# The 3D job: a 200 um straight section with a wave port at each end (J2 in the
# details). Only its mesh is built here; the cloud runs are in the details notebook.
DRIVEN_FMIN, DRIVEN_FMAX, DRIVEN_POINTS = 1e9, 40e9, 79  # Hz, Hz, 0.5 GHz steps
WAVE_MODE = 2  # port mode 2 is the line mode (mode 1 is the wall mode of the box)


class PECWallDrivenSim(DrivenSim):
    """A `DrivenSim` whose outer wall is a perfect conductor (J1's wall and the port solve's).

    `metallic_boundaries` is the switch that puts the outer wall under the PEC boundary
    condition; `BoundaryModeSim` has it as a field, `DrivenSim` only reads it if present.
    """

    metallic_boundaries: bool = True
    absorbing_boundary: bool = False


def driven_sim(length_um, refined_um, output_dir):
    """J2-J4: a straight section with a wave port at each end and a perfect-conductor outer wall."""
    comp, stack = build_cps(length_um, ports=True)
    sim = PECWallDrivenSim()
    sim.set_output_dir(output_dir)
    sim.set_geometry(comp)
    sim.set_stack(stack)
    sim.set_airbox(margin_x=0.0, margin_y=0.0, z_above=AIR_ABOVE, z_below=0.0)
    for port in ("o1", "o2"):
        sim.add_wave_port(
            port, layer="strip_s", max_size=True, mode=WAVE_MODE, excited=True
        )
    sim.set_driven(fmin=DRIVEN_FMIN, fmax=DRIVEN_FMAX, num_points=DRIVEN_POINTS)
    validation = sim.validate_config()
    assert validation.valid, validation.errors
    sim.mesh(
        preset="default",
        refined_mesh_size=refined_um,
        max_mesh_size=30.0,
        fmax=DRIVEN_FMAX,
        verbose=False,
    )
    sim.write_config()
    return sim


def mesh_counts(path):
    """``(tetrahedra, triangles)`` of a mesh file."""
    mesh = meshio.read(str(path))
    count = lambda kind: sum(len(c.data) for c in mesh.cells if c.type == kind)  # noqa: E731
    return count("tetra"), count("triangle")


sim_3d = driven_sim(200, 1.0, "./palace-sim-bj2012-j2")
tets, tris = mesh_counts(sim_3d.mesh_path)
print(f"3D mesh of the 200 um section: {tets} tetrahedra, {tris} boundary triangles")
groups = {
    name: dim
    for name, (_tag, dim) in meshio.read(
        sim_3d.mesh_path, file_format="gmsh"
    ).field_data.items()
}
metal_and_ports = [g for g in groups if g.startswith(("strip_", "P"))]
faint = [g for g, dim in groups.items() if dim == 2 and g not in metal_and_ports]
sim_3d.plot_mesh(style="solid", transparent_groups=faint, interactive=False)

## How it is simulated

1. **femwell, cross-section.** The 2D mesh above is solved for the propagating modes of the shielded line. The line mode gives its index, impedance and loss. This runs here, in about a minute.
2. **Palace, cross-section.** The same mesh goes to Palace's 2D boundary-mode solver: a second solver for the same quantities. The aluminum is either meshed as a volume of metal, or replaced by a surface condition on its outline (cheaper, but it ignores the field inside the metal).
3. **Palace, 3D.** Two straight sections, 200 µm and 600 µm long, with a wave port at each end. Removing the short one from the long one (a 2x-thru de-embedding) leaves a 400 µm line whose S-parameters give the propagation constant.
4. **Fit.** One passive line (R, L, C, G) is fitted to that propagation constant and to the impedance, which gives the numbers to compare with the paper's.

Steps 2 to 4 run in the cloud. This guide shows their stored results; the [details notebook](palace_bj2012_electrode_details.md) holds the code that submits and post-processes them and tests the post-processing on synthetic data.

<details>
<summary>The Palace jobs</summary>

| Job | What | Frequencies |
|---|---|---|
| J1 | 2D cross-section, strips as a surface condition (effective conductivity), line impedance across the gap | 1, 2.5, 5, 10, 20, 40 GHz |
| J1v | as J1 with the strips meshed as volumes of metal | same |
| J1b | as J1 (surface condition) with bulk aluminum | 10 GHz |
| J1vb | as J1v (meshed metal) with bulk aluminum | 10 GHz |
| J2 | 3D, 200 µm straight section, wave port at each end, perfect-conductor outer wall | 1 to 40 GHz, 79 points |
| J3 | 3D, 600 µm straight section, same wall | same |

</details>

In [ ]:
def solve_line_mode(sim, stack, model, f_hz, *, order=2, num_modes=6):
    """Solve the box modes; return ``(reading, mode, ranking)`` of the line mode.

    ``ranking`` lists every propagating mode by decreasing index as
    ``(number, n_eff, wall_mode)``: the number a Palace wave port gives it.
    """
    signal, ground = conductors(model)
    mesh = sim.mesh_path
    modes = solve_modes(
        mesh,
        epsilon=epsilon_by_region(mesh, stack, frequency_hz=f_hz),
        wavelength_um=speed_of_light / f_hz * 1e6,
        num_modes=num_modes,
        order=order,
        metallic_boundaries=True,
        n_guess=2.3,
    )
    readings = [
        (
            line_reading(
                m, frequency_hz=f_hz, mesh=mesh, signal=signal, return_=ground
            ),
            m,
        )
        for m in propagating_modes(list(modes))
    ]
    line = [rm for rm in readings if not rm[0].wall_mode]
    best = max(line, key=lambda rm: rm[0].n_eff.real)
    by_index = sorted(readings, key=lambda rm: -rm[0].n_eff.real)
    ranking = [
        (number, rm[0].n_eff, bool(rm[0].wall_mode))
        for number, rm in enumerate(by_index, start=1)
    ]
    return (*best, ranking)


# femwell on the mesh above, bulk aluminum, order-2 elements, 10 GHz.
_comp, stack_2d = build_cps(lossy_metal=True, sigma=SIGMA_AL_BULK)
reading, mode, _ranking = solve_line_mode(sim_2d, stack_2d, "volume", FREQ_HZ)
lp = line_params_from_neff([FREQ_HZ], [reading.n_eff], z0_ohm=[reading.z0_ohm])
alpha_db_cm = float(lp.alpha_rf_np_m[0]) * 8.686 / 100
z0 = complex(reading.z0_ohm)
print(
    f"n = {reading.n_eff.real:.4f}, Z = {z0.real:.2f}{z0.imag:+.2f}j ohm, loss = {alpha_db_cm:.3f} dB/cm"
)

# The stored values of this solve (femwell on the mesh that Palace's J1vb uses).
for key, now in (("fw-j1vb-alpha", alpha_db_cm), ("fw-j1vb-z0", z0)):
    stored = complex(ROWS[key]["value"], ROWS[key].get("value_imag") or 0.0)
    assert abs(now - stored) / abs(stored) <= 2e-3, (key, now, stored)
print("matches the stored rows fw-j1vb-alpha and fw-j1vb-z0 (rel. 2e-3)")

fig, ax = plt.subplots(figsize=(11, 2.6))
mode.plot_component("E", "y", part="abs", boundaries=False, colorbar=True, ax=ax)
for artist in ax.collections:  # dense field mesh: raster, not vector
    artist.set_rasterized(True)
ax.set(
    xlim=(-60, 60),
    ylim=(-5, 8),
    xlabel="y (um)",
    ylabel="z (um)",
    title="|Ey| of the line mode at 10 GHz (femwell)",
)
ax.set_aspect("equal")
plt.show()

Each bar divides a result at 10 GHz by the paper's value, so a bar at 1 is a match. The Palace bars are the stored cloud run; look at the resistance and the loss.

In [ ]:
QUANTITIES = (  # label, row-id suffix
    ("Re Z", "z0"),
    ("L", "L"),
    ("C", "C"),
    ("R", "R"),
    ("loss", "alpha"),
)
METHODS = (  # label, row-id prefix
    ("femwell, bulk aluminum", "fw-bulk"),
    ("femwell, fitted conductivity (10 MS/m)", "fw-eff"),
    ("Palace 3D, fitted conductivity (stored cloud run)", "pal-3d"),
)
fig, ax = plt.subplots(figsize=(9, 3.8))
width = 0.26
for k, (label, prefix) in enumerate(METHODS):
    ratios = [
        ROWS[f"{prefix}-{suffix}"]["value"]
        / ROWS[f"{prefix}-{suffix}"]["reference_value"]
        for _name, suffix in QUANTITIES
    ]
    ax.bar(np.arange(len(ratios)) + (k - 1) * width, ratios, width, label=label)
ax.axhline(1.0, color="k", lw=1)
ax.set_xticks(range(len(QUANTITIES)), [q[0] for q in QUANTITIES])
ax.set_ylabel("result / paper (10 GHz)")
ax.set_title("Metal-only coplanar strips against the paper")
ax.legend(fontsize=8, loc="lower right")
plt.show()

In [ ]:
def complex_of(row, key="value"):
    """The (complex) value or reference of a stored row."""
    return complex(row[key], row.get(f"{key}_imag") or 0.0)


def show(value):
    """A number for the table: complex values as a - bj."""
    if value.imag:
        return f"{value.real:.4g} {value.imag:+.2g}j"
    return f"{value.real:.4g}"


def agreement(row):
    """Relative error of a row against its reference (complex distance), or None."""
    if row.get("reference_value") is None:
        return None
    reference = complex_of(row, "reference_value")
    return abs(complex_of(row) - reference) / abs(reference)


def provenance(row):
    """Solver, version, order, mesh and hardware of a row; 'not recorded' if absent."""
    missing = "not recorded"
    return "; ".join(
        [
            f"{row['solver']} {row.get('solver_version') or missing}",
            f"order {row['element_order']}" if row.get("element_order") else missing,
            row.get("mesh") or missing,
            row.get("hardware") or missing,
        ]
    )


SHOWN = (
    "fw-bulk-alpha",
    "fw-bulk-R",
    "fw-bulk-z0",
    "fw-eff-alpha",
    "fw-j1vb-alpha",
    "pal-j1vb-alpha",
    "pal-j1vb-z0",
    "pal-j1vb-z0-cloud",
    "pal-j1b-alpha",
    "pal-3d-alpha",
    "pal-3d-z0",
    "pal-3d-L",
    "pal-3d-C",
    "pal-3d-R",
)
lines = [
    "| Quantity | Value | Reference | Difference | Role | Solver, order, mesh, hardware |",
    "|---|---|---|---|---|---|",
]
for key in SHOWN:
    row = ROWS[key]
    error = agreement(row)
    lines.append(
        f"| {row['quantity']} ({row['unit']}) {row['method']} "
        f"| {show(complex_of(row))} "
        f"| {show(complex_of(row, 'reference_value')) if error is not None else '-'}"
        f" {row.get('reference_label') or ''} "
        f"| {f'{error:.1%}' if error is not None else '-'} "
        f"| {row['role']} "
        f"| {provenance(row)} ({row['status']}) |"
    )
display(Markdown("\n".join(lines)))

## What agrees and what does not

- **Bulk aluminum gives half the measured loss.** 1.83 dB/cm against 3.8 dB/cm. The model's resistance is 2.19 k$\Omega$/m against the measured 4.9; the strips are 2.4 skin depths thick and the current crowds at the gap-facing edges, but that is not enough.
- **Palace and femwell agree in loss when the metal is meshed; the impedance is not settled.** On the same mesh, with multigrid off (`MGMaxLevels` 1, see [awslabs/palace#1177](https://github.com/awslabs/palace/issues/1177)), Palace gives 1.826 dB/cm and femwell 1.826 dB/cm: within 0.1 %. The earlier 2.00 dB/cm of runs with multigrid on came from that Palace issue, not from the model. For the impedance there are two Palace values. One, $57.26 - 2.19j\,\Omega$, is quoted from an earlier cloud run (its Palace version is not recorded) and is within 0.2 % of femwell's $57.34 - 2.19j\,\Omega$. The fresh cloud run, with a Palace build that predates the wave-port fix [awslabs/palace#1119](https://github.com/awslabs/palace/pull/1119) (merged 9 Nov), gives a real-valued $Z_{PV} = 56.07\,\Omega$, 2.3 % below femwell's $|Z|$ of 57.4 $\Omega$. The likely cause is the missing fix. A rerun with a Palace that includes it is pending.
- **The surface condition is 23 % short.** For bulk aluminum it gives 1.40 dB/cm against femwell's 1.83. The stored cloud run, with multigrid off for every 2D job (`mg_max_levels_2d` in its provenance file), gives 1.40 dB/cm as well, so the multigrid setting does not move this number.
- **The 3D agreement with the paper is a calibration.** The conductivity of 10 MS/m (a quarter of bulk) was chosen so that femwell's resistance equals the paper's. With it the 3D line is within 10 % of the paper in every quantity and the loss is 3.56 dB/cm against 3.8. That the fitted line agrees with the paper says the fit worked, not that the physics is predicted. A published sheet resistance for the aluminum film (Yang *et al.*, *Opt. Express* **22**(24), 29978) gives 2.54 dB/cm, about a third of the way to the measurement.

<details>
<summary>Definitions: three impedances and the port reference</summary>

A line's impedance depends on how it is defined. Three are used, all from the line mode's fields: $Z_{PI} = 2P/|I|^2$ (power over the current on the signal strip), $Z_{VI} = |V|/|I|$ and $Z_{PV} = |V|^2/2P$, with $V$ taken across the gap at mid-height of the strips. For the perfect-metal line at 10 GHz they are 58.3, 56.5 and 54.7 $\Omega$; on the lossy lines they agree within 1 %. The 3D fit uses $Z_{PV}$ as the impedance of the wave port's mode, because it leaves the capacitance $C = n/(cZ)$ unchanged when the metal loss is added; the other two definitions move the 3D line's $Z$, $L$ and $C$ by about ±3 %. The loss does not depend on the choice.

A wave port's S-parameters are normalised to its own mode, so the de-embedded $Z_c' = \sqrt{B/C}$ is not the line's impedance: $Z_c = Z_c' Z_p / 50$ with $Z_p$ the port mode's impedance from femwell. The derivation, the synthetic tests and the fit are in the [details notebook](palace_bj2012_electrode_details.md) (Steps 1 and 5 to 7).

</details>

## Limits

- The Palace numbers are from cloud runs. The Palace version (0dd8d015) is recorded for the stored results and the hardware and run time are not (the table says "not recorded"). That Palace predates #1119, so its $Z_{PV}$ is real-valued, and a rerun with a build that has the fix is pending. The cloud step that stores S-parameters and provenance is in the details notebook.
- The effective conductivity is one fitted number. It stands for everything the model leaves out (barrier layers, roughness, sloped sidewalls, the row of vias at the edge of the gap) and about two thirds of the gap to the measurement stays unexplained even with the published film.
- The box is a perfect conductor. All three solvers share it, so none is a check against an open device.
- The electrode dimensions are read from a figure, and four inputs are assumptions (above).

## Next

- [Phase shifter of a silicon modulator](tcad_bj2012_modulator.md): the junction that loads this line, simulated with DEVSIM and femwell.
- [RF electrode of a silicon modulator: simulation details](palace_bj2012_electrode_details.md): the long version.